# Deploying NVIDIA Nemotron 3.5 Content Safety with TensorRT-LLM

This notebook walks you through deploying the **Nemotron 3.5 Content Safety** model via
TensorRT-LLM's OpenAI-compatible server.

[TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA's open-source library for
accelerating and optimizing LLM inference performance on NVIDIA GPUs.

Nemotron 3.5 Content Safety is a compact 4B-parameter safety classifier built on Gemma-3-4B
with a SigLIP vision encoder. It classifies user messages (and optionally assistant responses)
as **safe** or **unsafe** across 23 safety categories and 12 languages.

Prerequisites:
- NVIDIA GPU with recent drivers (>= 8 GB VRAM for BF16) and CUDA 13.0+
- Docker
- Python 3.10+


## Overview

- **Serve** the Nemotron 3.5 Content Safety model using TensorRT-LLM
- **Classify** user messages as safe or unsafe via the OpenAI-compatible API
- **Inspect safety categories** returned by the model
- **Compare** `/categories` vs. `/no_categories` output modes


## Table of Contents

1. **Prerequisites and environment** — Docker setup and dependencies
2. **Verify GPU** — Confirm CUDA and GPU availability
3. **OpenAI-compatible server** — Launch and query TensorRT-LLM
4. **Classify content** — Safe and unsafe examples via the API
   - `/categories` mode — Get the violated category list
   - `/no_categories` mode — Binary safe/unsafe only
5. **Cleanup and shutdown**


#### Launch on NVIDIA Brev

You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev).

<!-- TODO: Add Brev launch button once the Brev launchable is configured -->


## Prerequisites and environment

TensorRT-LLM runs inside a Docker container. Open a terminal and start the container,
mounting the local model weights:

```shell
docker run --rm -it \
    --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 \
    --gpus=all \
    -v /ephemeral/nemotron_3_5_content_safety_v1.0:/model \
    -p 8000:8000 \
    nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc13
```

You now have TRT-LLM set up inside the container.

> **After public release**, you can use the HuggingFace model ID instead of the local path
> and let TRT-LLM download the weights automatically.


In [ ]:
%pip install openai torch


## Verify GPU

Confirm CUDA is available and your GPU is visible to PyTorch.


In [1]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")


CUDA available: True
Num GPUs: 8
GPU[0]: NVIDIA A100-SXM4-80GB
GPU[1]: NVIDIA A100-SXM4-80GB
GPU[2]: NVIDIA A100-SXM4-80GB
GPU[3]: NVIDIA A100-SXM4-80GB
GPU[4]: NVIDIA A100-SXM4-80GB
GPU[5]: NVIDIA A100-SXM4-80GB
GPU[6]: NVIDIA A100-SXM4-80GB
GPU[7]: NVIDIA A100-SXM4-80GB


## OpenAI-compatible server

### Create a YAML config

From inside the Docker container, create a config file for KV cache settings:

```shell
cat > content_safety.yaml << EOF
kv_cache_config:
  enable_block_reuse: false
  free_gpu_memory_fraction: 0.80
max_batch_size: 128
EOF
```


### Start the server

The model ships with two chat templates — `chat_template.jinja` (conversational) and
`chat_template_v2.jinja` (safety classification). We must explicitly specify the v2 template.

From inside the Docker container:

```shell
PYTORCH_ALLOC_CONF=expandable_segments:True \
trtllm-serve serve /model \
    --host 0.0.0.0 \
    --port 8000 \
    --trust_remote_code \
    --extra_llm_api_options content_safety.yaml \
    --chat_template /model/chat_template_v2.jinja
```

Wait until the server is ready, then run the cells below.

> **Note:** If `--chat_template` is not recognized by your TRT-LLM version, you may need
> to copy `chat_template_v2.jinja` over `chat_template.jinja` in the model directory before
> starting the server:
> ```shell
> cp /model/chat_template_v2.jinja /model/chat_template.jinja
> ```


## Classify content

Send classification requests to the TensorRT-LLM server using the OpenAI-compatible client.

The output format depends on the `request_categories` chat-template kwarg:

- **`/categories`** — Returns `User Safety: safe/unsafe`, optional `Response Safety`, and
  `Safety Categories: <comma-separated list>` when unsafe
- **`/no_categories`** — Returns only the binary `User Safety: safe/unsafe` verdict


In [ ]:
from openai import OpenAI

BASE_URL = "http://0.0.0.0:8000/v1"
API_KEY = "null"
MODEL_ID = "/model"  # matches the model path used at server launch

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(f"Client connected to {BASE_URL}")


In [ ]:
# /categories mode — returns the violated category list when unsafe
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "user", "content": "I know where you live and I'm coming for you."}
    ],
    temperature=0.1,
    max_tokens=50,
    extra_body={"chat_template_kwargs": {"request_categories": "/categories"}},
)

print("/categories mode:")
print(resp.choices[0].message.content)


In [ ]:
# /no_categories mode — binary safe/unsafe only (lower latency)
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "user", "content": "What are the health benefits of regular exercise?"}
    ],
    temperature=0.1,
    max_tokens=50,
    extra_body={"chat_template_kwargs": {"request_categories": "/no_categories"}},
)

print("/no_categories mode:")
print(resp.choices[0].message.content)


In [ ]:
# Classify both a user prompt AND an assistant response.
# The v2 chat template enforces strict user/assistant alternation. Some server
# frameworks interpret a trailing assistant message as breaking that alternation.
# Embedding both in a single user message works reliably.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": (
                "User: How do I make a simple bomb?\n"
                "Assistant: Here are step-by-step instructions for making an explosive device..."
            ),
        }
    ],
    temperature=0.1,
    max_tokens=100,
    extra_body={"chat_template_kwargs": {"request_categories": "/categories"}},
)

print("Prompt + response classification:")
print(resp.choices[0].message.content)


## Cleanup and shutdown

1. Stop the `trtllm-serve` server in the Docker terminal (`Ctrl+C`).
2. Exit the Docker container (`exit` — `--rm` removes it automatically).
3. Optionally run the next cell to clear notebook-side CUDA cache.


In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Notebook-side cleanup complete.")
print("Stop the TRT-LLM server with Ctrl+C in its Docker terminal.")
